
# Canonical site-specific EV, building, and PV preprocessing

This notebook is the single data-preparation entry point for the NTPLL and Center Hall SEGAN cases. It selects the reviewed meters/EVSEs, performs timestamp and sensor QC, reconstructs gross building demand from the original net meters, verifies

\[
p_{\mathrm{native\ net}}=p_{\mathrm{building,gross}}-p_{\mathrm{PV}},
\]

and writes model-ready inputs to `2025Data/Site_Data_2025`. It does not run an MPC model.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / '2025Data').exists():
    raise FileNotFoundError('Run this notebook from the UPSCALeDEV_2024 project root')
SITE_ROOT = ROOT / '2025Data/Site_Data_2025'
INPUT_DIR = SITE_ROOT
FIGURE_DIR = SITE_ROOT / 'figures'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BUILDING_RAW = ROOT / '2025Data/Bldg_data/Bldg_load.csv'
PV_RAW = ROOT / '2025Data/PV_data/CSV_2025-08-06-18-42-25.csv'
EV_QC_SOURCE = ROOT / '2025Data/EV_data/UCSD_AllSites_Merge_PostProcessedSession_QC.csv'
FORMAL_START = pd.Timestamp('2025-06-01 00:00:00')
FORMAL_END = pd.Timestamp('2025-07-31 23:45:00')
INTERVAL = pd.Timedelta(minutes=15)

SITE_CONFIG = {
    'NTPLL': {
        'load_columns': [
            'HOUSING_BLDG_2.MSBR2#Real Power Mean#kW',
            'HOUSING_BLDG_3.MSBR3#Real Power Mean#kW',
            'HOUSING_BLDG_4.MSBR4#Real Power Mean#kW',
            'HOUSING_BLDG_5.MSD5#Real Power Mean#kW',
            'HOUSING_BLDG_6.MSD6#Real Power Mean#kW',
            'HOUSING_BLDG_7.MSD7#Real Power Mean#kW',
            'HOUSING_BLDG_7.SS1#Real Power Mean#kW',
        ],
        'pv_columns': [
            'HOUSING_BLDG_2.PV_SYSTEM#Real Power Mean#kW',
            'HOUSING_BLDG_3.3HPV#Real Power Mean#kW',
            'HOUSING_BLDG_4.PV#Real Power Mean#kW',
        ],
        # 99.0--99.9 percentile observed-power proxy from the reviewed 2024 site study.
        'pv_capacity_proxy_kW': 55.33 + 54.88 + 56.72,
        'ev_site': 'UCSD Scholars Parking Structure',
        'evse_pattern': r'Scholars 1-(?:[1-9]|[12][0-9]|3[0-4])',
        'expected_evse': 34,
    },
    'Center_Hall': {
        'load_columns': ['University_Center.Center_Hall_E2451#Real Power Mean#kW'],
        'pv_columns': ['PV.Osler_Parking_PV_P8900#Real Power Mean#kW'],
        'pv_capacity_proxy_kW': 206.15,
        'ev_site': 'UCSD South Parking Structure',
        'evse_pattern': None,
        'expected_evse': 38,
    },
}
CHARGER_UNIT_LIMIT_KW = 1.664 / 0.25  # model/data envelope for each LiteON SC48 port
BESS_POWER_KW = 250.0
BESS_ENERGY_KWH = 332.0
print('Canonical site-data output root:', SITE_ROOT)


In [ ]:
def _read_meter_source(path):
    frame = pd.read_csv(path, low_memory=False)
    if 'Timestamp' not in frame.columns:
        raise KeyError(f'{path} is missing Timestamp')
    frame['Interval start'] = pd.to_datetime(frame['Timestamp'], errors='raise') - INTERVAL
    frame = frame.drop(columns='Timestamp').set_index('Interval start').sort_index()
    if frame.index.duplicated().any():
        raise ValueError(f'Duplicate meter timestamps in {path}')
    return frame.apply(pd.to_numeric, errors='coerce')

load_raw = _read_meter_source(BUILDING_RAW)
pv_raw = _read_meter_source(PV_RAW)

def _qc_selected_meters(raw, columns, resource):
    missing_columns = sorted(set(columns) - set(raw.columns))
    if missing_columns:
        raise KeyError(f'{resource} source columns missing: {missing_columns}')
    expected = pd.date_range(FORMAL_START, FORMAL_END, freq='15min')
    selected = raw.loc[:, columns].reindex(expected)
    inserted_timestamp = ~expected.isin(raw.index)
    missing_before = selected.isna()
    selected = selected.interpolate(method='time', limit=8, limit_area='inside')
    if selected.isna().any().any():
        bad = selected.index[selected.isna().any(axis=1)]
        raise ValueError(f'Unresolved {resource} gaps: {list(bad[:8])}')
    negative_before = selected < 0
    selected = selected.clip(lower=0.0)
    return selected, inserted_timestamp, missing_before.any(axis=1), negative_before.any(axis=1)

site_meter_outputs = {}
meter_summary_rows = []
for site, config in SITE_CONFIG.items():
    net_meters, load_inserted, load_imputed, load_clipped = _qc_selected_meters(
        load_raw, config['load_columns'], f'{site} building net meter'
    )
    pv_meters, pv_inserted, pv_imputed, pv_clipped = _qc_selected_meters(
        pv_raw, config['pv_columns'], f'{site} PV meter'
    )
    measured_net = net_meters.sum(axis=1)
    p_pv = pv_meters.sum(axis=1)
    p_load = measured_net + p_pv
    output = pd.DataFrame({
        'Interval start': measured_net.index,
        'p_load_kW': p_load.to_numpy(float),
        'p_PV_kW': p_pv.to_numpy(float),
        'p_native_net_kW': measured_net.to_numpy(float),
        'qc_inserted_timestamp': np.asarray(load_inserted | pv_inserted, dtype=bool),
        'qc_imputed_sensor_value': np.asarray(load_imputed | pv_imputed, dtype=bool),
        'qc_negative_sensor_clipped': np.asarray(load_clipped | pv_clipped, dtype=bool),
    })
    output['qc_any'] = output[[
        'qc_inserted_timestamp', 'qc_imputed_sensor_value', 'qc_negative_sensor_clipped'
    ]].any(axis=1)
    residual = output['p_native_net_kW'] - (output['p_load_kW'] - output['p_PV_kW'])
    max_residual = float(residual.abs().max())
    if max_residual > 1e-9:
        raise ValueError(f'{site} gross/net/PV identity failed: {max_residual:.3e} kW')
    if output[['p_load_kW', 'p_PV_kW']].isna().any().any():
        raise ValueError(f'{site} processed passive input contains NaN')
    out_path = INPUT_DIR / f'{site}_BTM_2025_June_July_15min_QC.csv'
    output.to_csv(out_path, index=False, float_format='%.6f')
    site_meter_outputs[site] = output
    meter_summary_rows.append({
        'site': site,
        'resource': 'passive_meter',
        'rows': len(output),
        'building_meter_count': len(config['load_columns']),
        'pv_meter_count': len(config['pv_columns']),
        'mean_gross_load_kW': float(output['p_load_kW'].mean()),
        'peak_gross_load_kW': float(output['p_load_kW'].max()),
        'mean_PV_kW': float(output['p_PV_kW'].mean()),
        'peak_PV_kW': float(output['p_PV_kW'].max()),
        'PV_observed_capacity_proxy_kW': float(config['pv_capacity_proxy_kW']),
        'June_peak_gross_load_kW': float(output.loc[output['Interval start'].between('2025-06-01', '2025-06-30 23:45:00'), 'p_load_kW'].max()),
        'June_peak_native_net_kW': float(output.loc[output['Interval start'].between('2025-06-01', '2025-06-30 23:45:00'), 'p_native_net_kW'].max()),
        'June_peak_PV_kW': float(output.loc[output['Interval start'].between('2025-06-01', '2025-06-30 23:45:00'), 'p_PV_kW'].max()),
        'minimum_native_net_kW': float(output['p_native_net_kW'].min()),
        'gross_net_identity_max_error_kW': max_residual,
        'qc_flagged_rows': int(output['qc_any'].sum()),
        'output_file': str(out_path.relative_to(ROOT)),
    })


In [ ]:
ev = pd.read_csv(EV_QC_SOURCE, low_memory=False)
for column in ['Interval start', 'Interval end', 'Session start', 'Session end']:
    ev[column] = pd.to_datetime(ev[column], errors='raise')

site_ev_outputs = {}
for site, config in SITE_CONFIG.items():
    selected = ev.loc[ev['Site'].eq(config['ev_site'])].copy()
    if config['evse_pattern'] is not None:
        selected = selected.loc[selected['Station Name'].astype(str).str.fullmatch(config['evse_pattern'])].copy()
    selected = selected.sort_values(['Interval start', 'Station Name', 'Session start']).reset_index(drop=True)
    observed_evse = int(selected['Station Name'].nunique(dropna=True))
    if observed_evse != config['expected_evse']:
        raise ValueError(f'{site}: expected {config["expected_evse"]} EVSE, found {observed_evse}')
    if site == 'NTPLL' and selected['Station Name'].astype(str).str.contains('Delta', case=False).any():
        raise ValueError('NTPLL filter incorrectly retained Delta chargers')
    if selected.empty or selected['Interval start'].duplicated().all():
        raise ValueError(f'{site} EV input is empty or malformed')
    out_path = INPUT_DIR / f'{site}_EV_2025_QC.csv'
    selected.to_csv(out_path, index=False)
    site_ev_outputs[site] = selected
    june = selected[selected['Interval start'].between('2025-06-01', '2025-06-30 23:45:00')]
    session_keys = ['Station Name', 'Session start']
    unique_sessions = june.drop_duplicates(session_keys)
    station_observed_peak = pd.to_numeric(selected['Interval max demand kW'], errors='coerce').groupby(selected['Station Name']).max()
    june_coincident_power = pd.to_numeric(june['Interval average demand kW'], errors='coerce').groupby(june['Interval start']).sum()
    meter_summary_rows.append({
        'site': site,
        'resource': 'EV',
        'rows': len(selected),
        'building_meter_count': np.nan,
        'pv_meter_count': np.nan,
        'configured_evse': config['expected_evse'],
        'observed_evse_full_period': observed_evse,
        'active_evse_June': int(june['Station Name'].nunique(dropna=True)),
        'June_sessions': len(unique_sessions),
        'June_session_energy_kWh': float(pd.to_numeric(unique_sessions['kWh delivered'], errors='coerce').sum()),
        'charger_model': 'LiteON SC48',
        'charger_unit_limit_kW': CHARGER_UNIT_LIMIT_KW,
        'installed_EVSE_power_kW': float(config['expected_evse'] * CHARGER_UNIT_LIMIT_KW),
        'observed_station_peak_sum_kW': float(station_observed_peak.sum()),
        'June_observed_coincident_EV_peak_kW': float(june_coincident_power.max()),
        'BESS_power_kW': BESS_POWER_KW,
        'BESS_energy_kWh': BESS_ENERGY_KWH,
        'output_file': str(out_path.relative_to(ROOT)),
    })

summary = pd.DataFrame(meter_summary_rows)
summary_path = INPUT_DIR / 'site_input_qc_summary.csv'
summary.to_csv(summary_path, index=False, float_format='%.6f')
display(summary)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, (site, btm) in zip(axes, site_meter_outputs.items()):
    june_btm = btm[btm['Interval start'].between('2025-06-01', '2025-06-30 23:45:00')].copy()
    june_btm['minute_of_day'] = june_btm['Interval start'].dt.hour * 60 + june_btm['Interval start'].dt.minute
    profile = june_btm.groupby('minute_of_day')[['p_load_kW', 'p_PV_kW', 'p_native_net_kW']].mean()
    x = profile.index.to_numpy() / 60.0
    ax.plot(x, profile['p_load_kW'], color='#4c78a8', lw=2.2, label='Gross building load')
    ax.plot(x, profile['p_PV_kW'], color='#f2a541', lw=2.2, label='PV generation')
    ax.plot(x, profile['p_native_net_kW'], color='#59a14f', lw=2.2, label='Measured native net load')
    ax.set_ylabel('Power (kW)')
    ax.set_title(site.replace('_', ' '), loc='left', fontweight='bold')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper center', ncol=3)
axes[-1].set_xlabel('Hour of day')
axes[-1].set_xticks(np.arange(0, 25, 2))
fig.suptitle('June 2025 Site Input Profiles after Meter QC', fontweight='bold', fontsize=16)
fig.tight_layout()
profile_figure = FIGURE_DIR / 'site_input_profiles_june_2025.png'
fig.savefig(profile_figure, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

for site, btm in site_meter_outputs.items():
    assert len(btm) == 61 * 96
    assert btm['Interval start'].iloc[0] == FORMAL_START
    assert btm['Interval start'].iloc[-1] == FORMAL_END
print('SITE INPUT VALIDATION: PASS')
print('Summary:', summary_path)
print('Figure:', profile_figure)
